<a href="https://colab.research.google.com/github/DenisPyankov/MT3-MAGENTA-colab/blob/denkuls/MT3_batch_plus_script_postprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MT3 batch + script postprocessing

<a href="https://colab.research.google.com/github/magenta/mt3/blob/main/mt3/colab/music_transcription_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music Transcription with Transformers

This notebook is an interactive demo of a few [music transcription models](g.co/magenta/mt3) created by Google's [Magenta](g.co/magenta) team.  You can upload audio and have one of our models automatically transcribe it.

<img src="https://magenta.tensorflow.org/assets/transcription-with-transformers/architecture_diagram.png" alt="Transformer-based transcription architecture">

The notebook supports two pre-trained models:
1. the piano transcription model from [our ISMIR 2021 paper](https://archives.ismir.net/ismir2021/paper/000030.pdf)
1. the multi-instrument transcription model from [our ICLR 2022 paper](https://openreview.net/pdf?id=iMSjopcOn0p)

**Caveat**: neither model is trained on singing.  If you upload audio with vocals, you will likely get weird results.  Multi-instrument transcription is still not a completely-solved problem and so you may get weird results regardless.

In any case, we hope you have fun transcribing!  Feel free to tweet any interesting output at [@GoogleMagenta](https://twitter.com/googlemagenta)...

### Instructions for running:

* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* Press ▶️ on the left of each cell to execute the cell
* In the __Load Model__ cell, choose either `ismir2021` for piano transcription or `mt3` for multi-instrument transcription
* In the __Upload Audio__ cell, choose an MP3 or WAV file from your computer when prompted
* Transcribe the audio using the __Transcribe Audio__ cell (it may take a few minutes depending on the length of the audio)

---

This notebook sends basic usage data to Google Analytics.  For more information, see [Google's privacy policy](https://policies.google.com/privacy).

In [ ]:
from google.colab import auth
auth.authenticate_user()


In [ ]:
# Copyright 2021 Google LLC. All Rights Reserved.

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

#@title Setup Environment
#@markdown Install MT3 and its dependencies (may take a few minutes).

!apt-get update -qq && apt-get install -qq libfluidsynth3 build-essential libasound2-dev libjack-dev

# install mt3
!git clone --branch=main https://github.com/magenta/mt3
!mv mt3 mt3_tmp; mv mt3_tmp/* .; rm -r mt3_tmp
!python3 -m pip install jax[cuda12] nest-asyncio pyfluidsynth==1.3.0 demucs -e . -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# copy checkpoints
!gcloud storage cp --recursive gs://mt3/checkpoints .

# copy soundfont (originally from https://sites.google.com/site/soundfonts4u)
!gcloud storage cp gs://magentadata/soundfonts/SGM-v2.01-Sal-Guit-Bass-V1.3.sf2 .

import json
import IPython

# The below functions (load_gtag and log_event) handle Google Analytics event
# logging. The logging is anonymous and stores only very basic statistics of the
# audio and transcription e.g. length of audio, number of transcribed notes.

def load_gtag():
  """Loads gtag.js."""
  # Note: gtag.js MUST be loaded in the same cell execution as the one doing
  # synthesis. It does NOT persist across cell executions!
  html_code = '''
<!-- Global site tag (gtag.js) - Google Analytics -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-4P250YRJ08"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());
  gtag('config', 'G-4P250YRJ08',
       {'referrer': document.referrer.split('?')[0],
        'anonymize_ip': true,
        'page_title': '',
        'page_referrer': '',
        'cookie_prefix': 'magenta',
        'cookie_domain': 'auto',
        'cookie_expires': 0,
        'cookie_flags': 'SameSite=None;Secure'});
</script>
'''
  IPython.display.display(IPython.display.HTML(html_code))

def log_event(event_name, event_details):
  """Log event with name and details dictionary."""
  details_json = json.dumps(event_details)
  js_string = "gtag('event', '%s', %s);" % (event_name, details_json)
  IPython.display.display(IPython.display.Javascript(js_string))

load_gtag()
log_event('setupComplete', {})

In [ ]:
# Фикс конфликтов
!pip uninstall -y tensorflow tensorflow-text jax jaxlib t5x
!pip install tensorflow==2.19.0 tensorflow-text==2.19.0
!pip install jax==0.4.13 jaxlib==0.4.13
!pip install git+https://github.com/google-research/t5x@94a89731086a3ee35e6aba182ce10d109424011c
!pip install git+https://github.com/magenta/note-seq


In [ ]:
#@title Imports and Definitions

import functools
import os

import numpy as np
import tensorflow.compat.v2 as tf

import functools
import gin
import jax
import librosa
import note_seq
import seqio
import t5
import t5x

from mt3 import metrics_utils
from mt3 import models
from mt3 import network
from mt3 import note_sequences
from mt3 import preprocessors
from mt3 import spectrograms
from mt3 import vocabularies

from google.colab import files

import nest_asyncio
nest_asyncio.apply()

SAMPLE_RATE = 16000
SF2_PATH = 'SGM-v2.01-Sal-Guit-Bass-V1.3.sf2'

def upload_audio(sample_rate):
  data = list(files.upload().values())
  if len(data) > 1:
    print('Multiple files uploaded; using only one.')
  return note_seq.audio_io.wav_data_to_samples_librosa(
    data[0], sample_rate=sample_rate)



class InferenceModel(object):
  """Wrapper of T5X model for music transcription."""

  def __init__(self, checkpoint_path, model_type='mt3'):

    # Model Constants.
    if model_type == 'ismir2021':
      num_velocity_bins = 127
      self.encoding_spec = note_sequences.NoteEncodingSpec
      self.inputs_length = 512
    elif model_type == 'mt3':
      num_velocity_bins = 1
      self.encoding_spec = note_sequences.NoteEncodingWithTiesSpec
      self.inputs_length = 256
    else:
      raise ValueError('unknown model_type: %s' % model_type)

    gin_files = ['/content/mt3/gin/model.gin',
                 f'/content/mt3/gin/{model_type}.gin']

    self.batch_size = 8
    self.outputs_length = 1024
    self.sequence_length = {'inputs': self.inputs_length,
                            'targets': self.outputs_length}

    self.partitioner = t5x.partitioning.PjitPartitioner(
        num_partitions=1)

    # Build Codecs and Vocabularies.
    self.spectrogram_config = spectrograms.SpectrogramConfig()
    self.codec = vocabularies.build_codec(
        vocab_config=vocabularies.VocabularyConfig(
            num_velocity_bins=num_velocity_bins))
    self.vocabulary = vocabularies.vocabulary_from_codec(self.codec)
    self.output_features = {
        'inputs': seqio.ContinuousFeature(dtype=tf.float32, rank=2),
        'targets': seqio.Feature(vocabulary=self.vocabulary),
    }

    # Create a T5X model.
    self._parse_gin(gin_files)
    self.model = self._load_model()

    # Restore from checkpoint.
    self.restore_from_checkpoint(checkpoint_path)

  @property
  def input_shapes(self):
    return {
          'encoder_input_tokens': (self.batch_size, self.inputs_length),
          'decoder_input_tokens': (self.batch_size, self.outputs_length)
    }

  def _parse_gin(self, gin_files):
    """Parse gin files used to train the model."""
    gin_bindings = [
        'from __gin__ import dynamic_registration',
        'from mt3 import vocabularies',
        'VOCAB_CONFIG=@vocabularies.VocabularyConfig()',
        'vocabularies.VocabularyConfig.num_velocity_bins=%NUM_VELOCITY_BINS'
    ]
    with gin.unlock_config():
      gin.parse_config_files_and_bindings(
          gin_files, gin_bindings, finalize_config=False)

  def _load_model(self):
    """Load up a T5X `Model` after parsing training gin config."""
    model_config = gin.get_configurable(network.T5Config)()
    module = network.Transformer(config=model_config)
    return models.ContinuousInputsEncoderDecoderModel(
        module=module,
        input_vocabulary=self.output_features['inputs'].vocabulary,
        output_vocabulary=self.output_features['targets'].vocabulary,
        optimizer_def=t5x.adafactor.Adafactor(decay_rate=0.8, step_offset=0),
        input_depth=spectrograms.input_depth(self.spectrogram_config))


  def restore_from_checkpoint(self, checkpoint_path):
    """Restore training state from checkpoint, resets self._predict_fn()."""
    train_state_initializer = t5x.utils.TrainStateInitializer(
      optimizer_def=self.model.optimizer_def,
      init_fn=self.model.get_initial_variables,
      input_shapes=self.input_shapes,
      partitioner=self.partitioner)

    restore_checkpoint_cfg = t5x.utils.RestoreCheckpointConfig(
        path=checkpoint_path, mode='specific', dtype='float32')

    train_state_axes = train_state_initializer.train_state_axes
    self._predict_fn = self._get_predict_fn(train_state_axes)
    self._train_state = train_state_initializer.from_checkpoint_or_scratch(
        [restore_checkpoint_cfg], init_rng=jax.random.PRNGKey(0))

  @functools.lru_cache()
  def _get_predict_fn(self, train_state_axes):
    """Generate a partitioned prediction function for decoding."""
    def partial_predict_fn(params, batch, decode_rng):
      return self.model.predict_batch_with_aux(
          params, batch, decoder_params={'decode_rng': None})
    return self.partitioner.partition(
        partial_predict_fn,
        in_axis_resources=(
            train_state_axes.params,
            t5x.partitioning.PartitionSpec('data',), None),
        out_axis_resources=t5x.partitioning.PartitionSpec('data',)
    )

  def predict_tokens(self, batch, seed=0):
    """Predict tokens from preprocessed dataset batch."""
    prediction, _ = self._predict_fn(
        self._train_state.params, batch, jax.random.PRNGKey(seed))
    return self.vocabulary.decode_tf(prediction).numpy()

  def __call__(self, audio):
    """Infer note sequence from audio samples.

    Args:
      audio: 1-d numpy array of audio samples (16kHz) for a single example.

    Returns:
      A note_sequence of the transcribed audio.
    """
    ds = self.audio_to_dataset(audio)
    ds = self.preprocess(ds)

    model_ds = self.model.FEATURE_CONVERTER_CLS(pack=False)(
        ds, task_feature_lengths=self.sequence_length)
    model_ds = model_ds.batch(self.batch_size)

    inferences = (tokens for batch in model_ds.as_numpy_iterator()
                  for tokens in self.predict_tokens(batch))

    predictions = []
    for example, tokens in zip(ds.as_numpy_iterator(), inferences):
      predictions.append(self.postprocess(tokens, example))

    result = metrics_utils.event_predictions_to_ns(
        predictions, codec=self.codec, encoding_spec=self.encoding_spec)
    return result['est_ns']

  def audio_to_dataset(self, audio):
    """Create a TF Dataset of spectrograms from input audio."""
    frames, frame_times = self._audio_to_frames(audio)
    return tf.data.Dataset.from_tensors({
        'inputs': frames,
        'input_times': frame_times,
    })

  def _audio_to_frames(self, audio):
    """Compute spectrogram frames from audio."""
    frame_size = self.spectrogram_config.hop_width
    padding = [0, frame_size - len(audio) % frame_size]
    audio = np.pad(audio, padding, mode='constant')
    frames = spectrograms.split_audio(audio, self.spectrogram_config)
    num_frames = len(audio) // frame_size
    times = np.arange(num_frames) / self.spectrogram_config.frames_per_second
    return frames, times

  def preprocess(self, ds):
    pp_chain = [
        functools.partial(
            t5.data.preprocessors.split_tokens_to_inputs_length,
            sequence_length=self.sequence_length,
            output_features=self.output_features,
            feature_key='inputs',
            additional_feature_keys=['input_times']),
        # Cache occurs here during training.
        preprocessors.add_dummy_targets,
        functools.partial(
            preprocessors.compute_spectrograms,
            spectrogram_config=self.spectrogram_config)
    ]
    for pp in pp_chain:
      ds = pp(ds)
    return ds

  def postprocess(self, tokens, example):
    tokens = self._trim_eos(tokens)
    start_time = example['input_times'][0]
    # Round down to nearest symbolic token step.
    start_time -= start_time % (1 / self.codec.steps_per_second)
    return {
        'est_tokens': tokens,
        'start_time': start_time,
        # Internal MT3 code expects raw inputs, not used here.
        'raw_inputs': []
    }

  @staticmethod
  def _trim_eos(tokens):
    tokens = np.array(tokens, np.int32)
    if vocabularies.DECODED_EOS_ID in tokens:
      tokens = tokens[:np.argmax(tokens == vocabularies.DECODED_EOS_ID)]
    return tokens



In [ ]:
import t5x.partitioning

def fixed_bounds_from_last_device(last_device):
    if hasattr(last_device, 'coords') and len(last_device.coords) == 3:
        x, y, z = last_device.coords
        # Для GPU атрибут core_on_chip может отсутствовать, используем 1
        core = getattr(last_device, 'core_on_chip', 1)
        return x + 1, y + 1, z + 1, core
    else:
        # Для не-TPU платформ (GPU) возвращаем единичные размеры
        return 1, 1, 1, 1

t5x.partitioning.bounds_from_last_device = fixed_bounds_from_last_device


In [ ]:
#@title Load Model
#@markdown The `ismir2021` model transcribes piano only, with note velocities.
#@markdown The `mt3` model transcribes multiple simultaneous instruments,
#@markdown but without velocities.

MODEL = "mt3" #@param["ismir2021", "mt3"]

checkpoint_path = f'/content/checkpoints/{MODEL}/'

load_gtag()

log_event('loadModelStart', {'event_category': MODEL})
inference_model = InferenceModel(checkpoint_path, MODEL)
log_event('loadModelComplete', {'event_category': MODEL})



# Пакетная обработка 5–10 аудиофайлов через Demucs + MT3

Этот блок заменяет одиночную загрузку файла на пакетный режим:

1. Берёт **папку на Google Drive** с аудио.
2. Делает **предобработку Demucs** для каждого файла.
3. Прогоняет каждый результат через **MT3**.
4. Сохраняет **raw MIDI от MT3** для дальнейшей постобработки.

> Ядро MT3 не меняется — меняется только внешний цикл, чтобы обработать несколько файлов подряд.


In [ ]:

from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import pandas as pd
from tqdm.auto import tqdm

drive.mount('/content/drive', force_remount=True)

# === Настройки ===
AUDIO_INPUT_DIR = Path('/content/drive/MyDrive/midi_job/piano')
WORK_ROOT = Path('/content/drive/MyDrive/midi_job/out')
STAGED_AUDIO_DIR = WORK_ROOT / 'audio_staged'
DEMUCS_OUTPUT_DIR = WORK_ROOT / 'demucs_output'
MT3_RAW_MIDI_DIR = WORK_ROOT / 'mt3_raw_midis'
MT3_REPORT_PATH = WORK_ROOT / 'mt3_transcription_report.csv'
PREPROC_REPORT_PATH = WORK_ROOT / 'preprocess_report.csv'

SUPPORTED_AUDIO_EXTS = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}
DEMUCS_MODEL = 'htdemucs'
MAX_FILES = 10

# === Подготовка папок ===
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

for d in [WORK_ROOT, STAGED_AUDIO_DIR, DEMUCS_OUTPUT_DIR, MT3_RAW_MIDI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert AUDIO_INPUT_DIR.exists(), f'Не найдена папка с аудио: {AUDIO_INPUT_DIR}'

# Берём только АУДИО, MIDI здесь игнорируются
audio_files_all = sorted(
    p for p in AUDIO_INPUT_DIR.rglob('*')
    if p.is_file() and p.name.lower() == 'normal.wav'
)

assert audio_files_all, f'В папке {AUDIO_INPUT_DIR} не найдено файлов normal.wav.'

audio_files = audio_files_all[:MAX_FILES]

print(f'Всего найдено папок с normal.wav: {len(audio_files_all)}')
print(f'В обработку взято первых: {len(audio_files)}')
for p in audio_files:
    print(' -', p.relative_to(AUDIO_INPUT_DIR))

# === Стадируем файлы с уникальными именами, чтобы Demucs ничего не перезаписал ===
manifest_rows = []
for idx, src_path in enumerate(audio_files, start=1):
    safe_name = f'{idx:02d}__{src_path.stem}{src_path.suffix.lower()}'
    staged_path = STAGED_AUDIO_DIR / safe_name
    shutil.copy2(src_path, staged_path)

    manifest_rows.append({
        'index': idx,
        'input_rel': str(src_path.relative_to(AUDIO_INPUT_DIR)),
        'input_path': str(src_path),
        'staged_name': safe_name,
        'staged_path': str(staged_path),
        'instrumental_path': '',
        'preprocess_status': None,
        'preprocess_error': None,
    })

manifest_df = pd.DataFrame(manifest_rows)

# === Demucs по каждому файлу ===
for i, row in tqdm(list(manifest_df.iterrows()), total=len(manifest_df), desc='Demucs preprocessing'):
    staged_path = Path(row['staged_path'])
    staged_stem = staged_path.stem

    try:
        subprocess.run([
            'demucs',
            '-n', DEMUCS_MODEL,
            '--two-stems=vocals',
            '--shifts=2',
            '--overlap=0.5',
            '--out', str(DEMUCS_OUTPUT_DIR),
            str(staged_path),
        ], check=True)

        instrumental_path = DEMUCS_OUTPUT_DIR / DEMUCS_MODEL / staged_stem / 'no_vocals.wav'
        if not instrumental_path.exists():
            raise FileNotFoundError(f'Не найден результат Demucs: {instrumental_path}')

        manifest_df.at[i, 'instrumental_path'] = str(instrumental_path)
        manifest_df.at[i, 'preprocess_status'] = 'ok'
    except Exception as e:
        manifest_df.at[i, 'preprocess_status'] = 'error'
        manifest_df.at[i, 'preprocess_error'] = f'{type(e).__name__}: {e}'

manifest_df.to_csv(PREPROC_REPORT_PATH, index=False)

print('\nСтатусы предобработки:')
display(manifest_df[['input_rel', 'preprocess_status', 'preprocess_error', 'instrumental_path']])
print('\nСохранён отчёт:', PREPROC_REPORT_PATH)


In [ ]:

#@title Transcribe Batch Audio with MT3

import librosa
import note_seq
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

load_gtag()

def normalize_mt3_output_to_single_piano(est_ns):
    min_duration_sec = 0.0
    cleaned_notes = []

    for note in est_ns.notes:
        duration = note.end_time - note.start_time
        if duration >= min_duration_sec:
            note.program = 0
            note.instrument = 0
            cleaned_notes.append(note)

    del est_ns.notes[:]
    est_ns.notes.extend(cleaned_notes)
    return est_ns

mt3_rows = []

ok_manifest = manifest_df[manifest_df['preprocess_status'] == 'ok'].copy()
assert len(ok_manifest) > 0, 'После Demucs не осталось файлов со статусом ok.'

for _, row in tqdm(ok_manifest.iterrows(), total=len(ok_manifest), desc='MT3 transcription'):
    input_rel = row['input_rel']
    instrumental_path = Path(row['instrumental_path'])
    rel_parent = Path(input_rel).parent
    out_dir = MT3_RAW_MIDI_DIR / rel_parent
    out_dir.mkdir(parents=True, exist_ok=True)

    out_midi = out_dir / f'{Path(input_rel).stem}_mt3_raw.mid'

    report_row = {
        'input_rel': input_rel,
        'instrumental_path': str(instrumental_path),
        'raw_midi_path': str(out_midi),
        'duration_sec': None,
        'note_count': None,
        'status': None,
        'error': None,
    }

    try:
        log_event('transcribeStart', {
            'event_category': MODEL,
            'input_rel': input_rel,
        })

        audio, _ = librosa.load(str(instrumental_path), sr=SAMPLE_RATE)
        report_row['duration_sec'] = round(len(audio) / SAMPLE_RATE, 3)

        est_ns = inference_model(audio)
        est_ns = normalize_mt3_output_to_single_piano(est_ns)

        note_seq.sequence_proto_to_midi_file(est_ns, str(out_midi))
        report_row['note_count'] = sum(1 for note in est_ns.notes if not note.is_drum)
        report_row['status'] = 'ok'

        log_event('transcribeComplete', {
            'event_category': MODEL,
            'value': round(len(audio) / SAMPLE_RATE),
            'numNotes': report_row['note_count'],
        })
    except Exception as e:
        report_row['status'] = 'error'
        report_row['error'] = f'{type(e).__name__}: {e}'

    mt3_rows.append(report_row)

mt3_report_df = pd.DataFrame(mt3_rows)
mt3_report_df.to_csv(MT3_REPORT_PATH, index=False)

raw_midi_files = sorted(MT3_RAW_MIDI_DIR.rglob('*.mid'))

print('\nСтатусы MT3:')
display(mt3_report_df)
print(f'\nСохранено raw MIDI от MT3: {len(raw_midi_files)}')
print('Папка raw MIDI:', MT3_RAW_MIDI_DIR)
print('Отчёт MT3:', MT3_REPORT_PATH)



# Постобработка через скриптовый cleaner

Ниже добавлен второй этап:

- вход: **raw MIDI, полученные из MT3**
- постобработка: **скриптовая очистка MIDI**
- выход:
  - `processed_midi/`
  - `report.csv`

Этот вариант легче, быстрее и детерминированнее, чем AI-постобработка.


## Установка зависимостей для скриптовой очистки

In [ ]:
%pip -q install "symusic>=0.5.9" "miditok>=3.0.0" "pretty_midi>=0.2.10" "mido>=1.3.0" pandas

## Импорты

In [ ]:
from __future__ import annotations

import inspect
import math
import tempfile
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd
import pretty_midi
import mido

try:
    from symusic import Score
    HAS_SYMUSIC = True
except Exception:
    HAS_SYMUSIC = False

try:
    from miditok import REMI, TokenizerConfig
    HAS_MIDITOK = True
except Exception:
    HAS_MIDITOK = False

print("HAS_SYMUSIC =", HAS_SYMUSIC)
print("HAS_MIDITOK =", HAS_MIDITOK)

In [ ]:

#@title Script postprocessing paths and setup

from pathlib import Path
import shutil

SCRIPT_OUTPUT_DIR = WORK_ROOT / 'script_postprocess' / 'processed_midi'
SCRIPT_REPORT_PATH = WORK_ROOT / 'script_postprocess' / 'report.csv'

script_root = SCRIPT_OUTPUT_DIR.parent
if script_root.exists():
    shutil.rmtree(script_root)
SCRIPT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_midis = sorted(list(MT3_RAW_MIDI_DIR.rglob('*.mid')) + list(MT3_RAW_MIDI_DIR.rglob('*.midi')))
assert raw_midis, f'В папке {MT3_RAW_MIDI_DIR} не найдено raw MIDI от MT3.'

print('MT3 raw MIDI files =', len(raw_midis))
print('SCRIPT_OUTPUT_DIR =', SCRIPT_OUTPUT_DIR)
print('SCRIPT_REPORT_PATH =', SCRIPT_REPORT_PATH)


## Safe load MIDI

In [ ]:
def _score_supports_sanitize() -> bool:
    if not HAS_SYMUSIC:
        return False
    try:
        sig = inspect.signature(Score)
        return "sanitize_data" in sig.parameters
    except Exception:
        return False


def try_symusic_load(path: Path):
    if not HAS_SYMUSIC:
        return None, "symusic_not_installed"
    kwargs = {"ttype": "tick"}
    if _score_supports_sanitize():
        kwargs["sanitize_data"] = True
    try:
        score = Score(str(path), **kwargs)
        return score, "symusic"
    except TypeError:
        try:
            score = Score(str(path), ttype="tick")
            return score, "symusic_no_sanitize"
        except Exception as e:
            return None, f"symusic_failed: {type(e).__name__}: {e}"
    except Exception as e:
        return None, f"symusic_failed: {type(e).__name__}: {e}"


def load_pretty_midi_best_effort(path: Path):
    errors = []

    try:
        return pretty_midi.PrettyMIDI(str(path)), "pretty_midi_direct", errors
    except Exception as e:
        errors.append(f"pretty_midi_direct: {type(e).__name__}: {e}")

    try:
        midi = mido.MidiFile(str(path), clip=True)
        with tempfile.NamedTemporaryFile(suffix=".mid", delete=False) as tmp:
            tmp_path = Path(tmp.name)
        midi.save(str(tmp_path))
        pm = pretty_midi.PrettyMIDI(str(tmp_path))
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass
        return pm, "mido_clip_then_pretty_midi", errors
    except Exception as e:
        errors.append(f"mido_clip_then_pretty_midi: {type(e).__name__}: {e}")

    raise RuntimeError("Не удалось прочитать MIDI. Ошибки: " + " | ".join(errors))

Задаём константы для cleaner

In [ ]:
from dataclasses import dataclass
import math

# --- настройки очистки MIDI ---
QUANTIZE_STEP_SEC = 0.05      # шаг квантования в секундах
MIN_DURATION_SEC = 0.03       # минимальная длительность ноты
DEDUP_TOL_SEC = 0.02          # допуск для удаления дублей по onset
OVERLAP_TOL_SEC = 0.01        # допуск для overlap одинаковой высоты
DROP_EMPTY_INSTRUMENTS = True # удалять пустые инструменты

## Cleaner

In [ ]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


@dataclass
class CleanStats:
    file: str
    load_backend: str
    symusic_backend: str | None = None
    instruments_total: int = 0
    instruments_dropped_empty: int = 0
    notes_before: int = 0
    notes_after: int = 0
    notes_removed_duplicate: int = 0
    notes_fixed_non_positive_duration: int = 0
    notes_extended_short: int = 0
    notes_trimmed_same_pitch_overlap: int = 0
    notes_removed_invalid: int = 0
    pitches_clamped: int = 0
    velocities_clamped: int = 0
    control_changes_clamped: int = 0
    control_changes_removed_duplicate: int = 0
    pitch_bends_clamped: int = 0
    quantized_events: int = 0
    read_error: str | None = None


def _quantize(x: float, step: float | None) -> float:
    if not step or step <= 0:
        return x
    return round(x / step) * step


def _clean_control_changes(inst, stats: CleanStats, tol: float = 1e-6):
    cleaned = []
    seen = set()
    for cc in sorted(inst.control_changes, key=lambda x: (float(x.time), int(x.number), int(x.value))):
        t = float(cc.time)
        number = clamp(int(cc.number), 0, 127)
        value = clamp(int(cc.value), 0, 127)
        if number != int(cc.number) or value != int(cc.value):
            stats.control_changes_clamped += 1
        if QUANTIZE_STEP_SEC:
            qt = _quantize(t, QUANTIZE_STEP_SEC)
            if abs(qt - t) > tol:
                stats.quantized_events += 1
            t = qt
        key = (round(t / max(tol, 1e-9)), number, value)
        if key in seen:
            stats.control_changes_removed_duplicate += 1
            continue
        seen.add(key)
        cleaned.append(pretty_midi.ControlChange(number=number, value=value, time=t))
    inst.control_changes = cleaned


def _clean_pitch_bends(inst, stats: CleanStats, tol: float = 1e-6):
    cleaned = []
    seen = set()
    for pb in sorted(inst.pitch_bends, key=lambda x: (float(x.time), int(x.pitch))):
        t = float(pb.time)
        pitch = clamp(int(pb.pitch), -8192, 8191)
        if pitch != int(pb.pitch):
            stats.pitch_bends_clamped += 1
        if QUANTIZE_STEP_SEC:
            qt = _quantize(t, QUANTIZE_STEP_SEC)
            if abs(qt - t) > tol:
                stats.quantized_events += 1
            t = qt
        key = (round(t / max(tol, 1e-9)), pitch)
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(pretty_midi.PitchBend(pitch=pitch, time=t))
    inst.pitch_bends = cleaned


def _prepare_note(note, stats: CleanStats):
    try:
        start = float(note.start)
        end = float(note.end)
        pitch = int(note.pitch)
        velocity = int(note.velocity)
    except Exception:
        stats.notes_removed_invalid += 1
        return None

    if any(map(math.isnan, [start, end])):
        stats.notes_removed_invalid += 1
        return None

    pitch2 = clamp(pitch, 0, 127)
    if pitch2 != pitch:
        stats.pitches_clamped += 1
        pitch = pitch2

    velocity2 = clamp(velocity, 1, 127)
    if velocity2 != velocity:
        stats.velocities_clamped += 1
        velocity = velocity2

    if QUANTIZE_STEP_SEC:
        qstart = _quantize(start, QUANTIZE_STEP_SEC)
        qend = _quantize(end, QUANTIZE_STEP_SEC)
        start, end = qstart, qend

    if end <= start:
        end = start + MIN_DURATION_SEC
        stats.notes_fixed_non_positive_duration += 1

    if (end - start) < MIN_DURATION_SEC:
        end = start + MIN_DURATION_SEC
        stats.notes_extended_short += 1

    return {
        "start": start,
        "end": end,
        "pitch": pitch,
        "velocity": velocity,
    }


def _deduplicate_notes(prepared_notes, stats: CleanStats):
    groups = {}
    for n in prepared_notes:
        onset_bucket = round(n["start"] / DEDUP_TOL_SEC)
        key = (n["pitch"], onset_bucket)
        prev = groups.get(key)
        if prev is None:
            groups[key] = n
        else:
            prev_score = (prev["end"] - prev["start"], prev["velocity"])
            cur_score = (n["end"] - n["start"], n["velocity"])
            if cur_score > prev_score:
                groups[key] = n
            stats.notes_removed_duplicate += 1
    return sorted(groups.values(), key=lambda x: (x["start"], x["pitch"], x["end"], x["velocity"]))


def _trim_same_pitch_overlaps(notes, stats: CleanStats, is_drum: bool):
    if is_drum:
        return notes

    cleaned = []
    last_idx_by_pitch = {}

    for n in notes:
        pitch = n["pitch"]
        if pitch in last_idx_by_pitch:
            prev = cleaned[last_idx_by_pitch[pitch]]
            if n["start"] < prev["end"] - OVERLAP_TOL_SEC:
                new_prev_end = max(prev["start"] + MIN_DURATION_SEC, n["start"])
                if new_prev_end < prev["end"] - OVERLAP_TOL_SEC:
                    prev["end"] = new_prev_end
                    stats.notes_trimmed_same_pitch_overlap += 1
                elif n["end"] <= prev["end"] + OVERLAP_TOL_SEC:
                    stats.notes_removed_invalid += 1
                    continue
        cleaned.append(n)
        last_idx_by_pitch[pitch] = len(cleaned) - 1

    return cleaned


def clean_pretty_midi(pm: pretty_midi.PrettyMIDI, file_name: str, load_backend: str, symusic_backend: str | None):
    stats = CleanStats(file=file_name, load_backend=load_backend, symusic_backend=symusic_backend)
    stats.instruments_total = len(pm.instruments)

    for inst in pm.instruments:
        stats.notes_before += len(inst.notes)

        prepared = []
        for note in sorted(inst.notes, key=lambda x: (float(x.start), int(x.pitch), float(x.end), int(x.velocity))):
            p = _prepare_note(note, stats)
            if p is not None:
                prepared.append(p)

        prepared = _deduplicate_notes(prepared, stats)
        prepared = _trim_same_pitch_overlaps(prepared, stats, is_drum=inst.is_drum)

        inst.notes = [
            pretty_midi.Note(
                velocity=n["velocity"],
                pitch=n["pitch"],
                start=n["start"],
                end=n["end"],
            )
            for n in prepared
        ]

        _clean_control_changes(inst, stats)
        _clean_pitch_bends(inst, stats)

    if DROP_EMPTY_INSTRUMENTS:
        kept = []
        for inst in pm.instruments:
            if inst.notes or inst.control_changes or inst.pitch_bends:
                kept.append(inst)
            else:
                stats.instruments_dropped_empty += 1
        pm.instruments = kept

    stats.notes_after = sum(len(inst.notes) for inst in pm.instruments)
    return pm, stats

## Обработка одного MIDI

In [ ]:

def clean_one_midi(path: str | Path, output_dir: str | Path = SCRIPT_OUTPUT_DIR, input_root: str | Path = MT3_RAW_MIDI_DIR):
    path = Path(path)
    output_dir = Path(output_dir)
    input_root = Path(input_root)

    output_dir.mkdir(parents=True, exist_ok=True)

    symusic_backend = None
    if HAS_SYMUSIC:
        _, symusic_backend = try_symusic_load(path)

    pm, load_backend, errors = load_pretty_midi_best_effort(path)
    pm, stats = clean_pretty_midi(
        pm,
        file_name=path.name,
        load_backend=load_backend,
        symusic_backend=symusic_backend
    )

    rel_parent = path.relative_to(input_root).parent
    target_dir = output_dir / rel_parent
    target_dir.mkdir(parents=True, exist_ok=True)

    out_path = target_dir / f"{path.stem}_script_clean.mid"
    pm.write(str(out_path))

    meta = asdict(stats)
    meta["source_path"] = str(path)
    meta["output_path"] = str(out_path)
    meta["load_errors"] = " | ".join(errors) if errors else ""
    return out_path, meta


## Батч script-постобработки по raw MIDI от MT3

In [ ]:

#@title Run script postprocessing on MT3 raw MIDI folder

all_files = sorted(list(MT3_RAW_MIDI_DIR.rglob("*.mid")) + list(MT3_RAW_MIDI_DIR.rglob("*.midi")))

rows = []
for path in all_files:
    try:
        out_path, meta = clean_one_midi(path, SCRIPT_OUTPUT_DIR, MT3_RAW_MIDI_DIR)
        rows.append(meta)
    except Exception as e:
        rows.append({
            "file": path.name,
            "source_path": str(path),
            "output_path": "",
            "error": f"{type(e).__name__}: {e}",
        })

script_report_df = pd.DataFrame(rows)
script_report_df.to_csv(SCRIPT_REPORT_PATH, index=False)

print("Files processed:", len(script_report_df))
print("Report saved to:", SCRIPT_REPORT_PATH.resolve())
display(script_report_df.head(20))



## Как сравнивать script vs AI после прогона

Сравнение корректно делать только на **одних и тех же raw MIDI от MT3**.

На практике:

- **Script** — лучший выбор, когда нужна мягкая и предсказуемая чистка без больших музыкальных допущений.
- **AI** — лучший выбор, когда важнее получить более «нотно-осмысленный» результат и MusicXML.

Для честного вывода запустите оба ноутбука на одной папке и сравните итоговые `processed_midi/` и отчёты.


In [ ]:

#@title Export script-postprocessing results

import shutil
import zipfile
from pathlib import Path
from google.colab import files

SCRIPT_EXPORT_ROOT = WORK_ROOT / 'export_script'
SCRIPT_RESULT_ZIP = Path('/content/drive/MyDrive/midi_job/out/mt3_script_postprocess_result.zip')

if SCRIPT_EXPORT_ROOT.exists():
    shutil.rmtree(SCRIPT_EXPORT_ROOT)
SCRIPT_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

# Копируем отчёты
shutil.copy2(PREPROC_REPORT_PATH, SCRIPT_EXPORT_ROOT / PREPROC_REPORT_PATH.name)
shutil.copy2(MT3_REPORT_PATH, SCRIPT_EXPORT_ROOT / MT3_REPORT_PATH.name)
shutil.copy2(SCRIPT_REPORT_PATH, SCRIPT_EXPORT_ROOT / SCRIPT_REPORT_PATH.name)

# Копируем raw MIDI от MT3
raw_export_dir = SCRIPT_EXPORT_ROOT / 'raw_mt3_midis'
shutil.copytree(MT3_RAW_MIDI_DIR, raw_export_dir, dirs_exist_ok=True)

# Копируем результаты скриптовой постобработки
clean_export_dir = SCRIPT_EXPORT_ROOT / 'processed_midi'
shutil.copytree(SCRIPT_OUTPUT_DIR, clean_export_dir, dirs_exist_ok=True)

if SCRIPT_RESULT_ZIP.exists():
    SCRIPT_RESULT_ZIP.unlink()

with zipfile.ZipFile(SCRIPT_RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in SCRIPT_EXPORT_ROOT.rglob('*'):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(SCRIPT_EXPORT_ROOT)))

print('Готов архив:', SCRIPT_RESULT_ZIP)
files.download(str(SCRIPT_RESULT_ZIP))
